In [1]:
import os
import pandas as pd

In [ ]:
#This notebook takes the input csv files after filtering and parses the information for each parameter into a single csv file, with the column headers the condition + replicate of interest.
#It can take this parsed information and generate two types of output, a file with every value for individual replicates for a given parameter, or a second set of csvs where all the replicates for a given condition
#are merged. In each case summary csv files with mean and median values will also be generated.

#To run the script, pass a list of parameters to the function.

def parse_and_merge_csvs(parameters_list):
    # Define the base folder to analyze all subfolders within it. Base folder contains the csv files for each condition after filtering.
    base_folder = "/your_input_folder/output_filtered/"
    individual_output_base_folder = "output_folder/individual_cells/"
    merged_output_base_folder = "output_folder/individual_cells/"

    # Ensure the base output folders exist or create them
    os.makedirs(individual_output_base_folder, exist_ok=True)
    os.makedirs(merged_output_base_folder, exist_ok=True)

    # Walk through all subfolders in the base folder
    for root, dirs, files in os.walk(base_folder):
        subfolder_name = os.path.basename(root)  # Get the name of each subfolder
        
        if not files:  # Skip empty folders
            continue

        # Define subfolder-specific output directories within each threshold subfolder
        individual_output_dir = os.path.join(individual_output_base_folder, subfolder_name, "individual_replicates")
        merged_output_dir = os.path.join(merged_output_base_folder, subfolder_name, "merged_replicates")
        summary_output_dir = os.path.join(merged_output_base_folder, subfolder_name, "summary")

        # Define subdirectories for mean and median CSVs within the summary folder
        mean_output_dir = os.path.join(summary_output_dir, 'mean')
        median_output_dir = os.path.join(summary_output_dir, 'median')

        # Create the subdirectories if they don't exist
        os.makedirs(mean_output_dir, exist_ok=True)
        os.makedirs(median_output_dir, exist_ok=True)
        os.makedirs(individual_output_dir, exist_ok=True)
        os.makedirs(merged_output_dir, exist_ok=True)

        # Dictionary to store dataframes for each parameter
        parameter_dataframes = {parameter: [] for parameter in parameters_list}

        for filename in files:
            if filename.endswith('.csv'):
                filepath = os.path.join(root, filename)
                df = pd.read_csv(filepath)

                # Extract condition and replicate information
                condition = df['condition'].unique()[0]
                replicate = df['replicate'].unique()[0]

                # Filter only rep1, rep2, and rep3. Add additional replicates as necessary.
                if replicate not in ['rep1', 'rep2', 'rep3']:
                    continue

                # Process each specified parameter
                for parameter in parameters_list:
                    new_column_name = f'{condition}_{replicate}'
                    parsed_df = pd.DataFrame({new_column_name: df[parameter]})
                    parameter_dataframes[parameter].append(parsed_df)

        # Process each parameter to concatenate and save mean/median summaries
        for parameter, dataframes in parameter_dataframes.items():
            final_df = pd.concat(dataframes, axis=1)
            final_df = final_df[sorted(final_df.columns)]

            # Save merged data for individual replicates
            output_filepath = os.path.join(individual_output_dir, f'merged_{parameter}_individual_replicates.csv')
            final_df.to_csv(output_filepath, index=False)
            print(f"Saved individual replicates merged data for '{parameter}' to {output_filepath}")
            
            # Initialize dictionaries to store mean and median values
            mean_dict = {}
            median_dict = {}

            # Group replicates under the same condition and calculate mean/median
            for col in final_df.columns:
                condition_name = '_'.join(col.split('_')[:-1])

                mean_val = final_df[col].mean()
                median_val = final_df[col].median()

                if condition_name not in mean_dict:
                    mean_dict[condition_name] = []
                    median_dict[condition_name] = []

                mean_dict[condition_name].append(mean_val)
                median_dict[condition_name].append(median_val)

            mean_df = pd.DataFrame(dict([(k, pd.Series(v)) for k, v in mean_dict.items()]))
            median_df = pd.DataFrame(dict([(k, pd.Series(v)) for k, v in median_dict.items()]))

            mean_output_filepath = os.path.join(mean_output_dir, f'mean_{parameter}_summary.csv')
            median_output_filepath = os.path.join(median_output_dir, f'median_{parameter}_summary.csv')

            mean_df.to_csv(mean_output_filepath, index=False)
            median_df.to_csv(median_output_filepath, index=False)
            print(f"Saved mean summary for '{parameter}' to {mean_output_filepath}")
            print(f"Saved median summary for '{parameter}' to {median_output_filepath}")

            # Combine replicates into a single column for unique conditions
            unique_condition_data = {}
            for col in final_df.columns:
                condition_name = '_'.join(col.split('_')[:-1])

                if condition_name not in unique_condition_data:
                    unique_condition_data[condition_name] = []
                unique_condition_data[condition_name].extend(final_df[col].tolist())

            unique_condition_df = pd.DataFrame(dict([(k, pd.Series(v)) for k, v in unique_condition_data.items()]))
            unique_condition_output_filepath = os.path.join(merged_output_dir, f'merged_{parameter}_all_replicates.csv')
            unique_condition_df.to_csv(unique_condition_output_filepath, index=False)
            print(f"Saved merged data for unique conditions for '{parameter}' to {unique_condition_output_filepath}")


In [ ]:
#specify the parameters from the input csv that you want to parse to generate the input file for plotting
parameters_list = ['mean_kat2b', 'mean_hoechst', 'nuclear_area_microns', 'mean_h3s10p'] 
parse_and_merge_csvs(parameters_list)